# 01 - Preprocess FIPS Codes for CRD Mortality Data

This notebook merges the CRD mortality data with FIPS codes to enable joining with other datasets (ACS, weather, livestock).

The location names in IHME data are in the format "County Name (State)", which we parse to extract County and State, then merge with the FIPS lookup table.

In [ ]:
import pandas as pd
import os

In [ ]:
def preprocess_fips_crd(year):
    fips_path = '../data/raw/state_fips.csv'
    crd_path = f'../data/processed/crd_single_year/crd_mortality_{year}.csv'

    fips_df = pd.read_csv(fips_path, dtype={'fips': str})
    fips_df['fips'] = fips_df['fips'].str.zfill(5)
    fips_df['State_FIPS'] = fips_df['fips'].str[:2]
    fips_df['County_FIPS'] = fips_df['fips'].str[2:]
    fips_df['state_full'] = fips_df['state'].map({
        'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas',
        'CA': 'California', 'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware',
        'FL': 'Florida', 'GA': 'Georgia', 'HI': 'Hawaii', 'ID': 'Idaho',
        'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa', 'KS': 'Kansas',
        'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
        'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi',
        'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada',
        'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York',
        'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma',
        'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
        'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah',
        'VT': 'Vermont', 'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia',
        'WI': 'Wisconsin', 'WY': 'Wyoming'
    })

    crd_df = pd.read_csv(crd_path)

    crd_df[['County', 'State']] = crd_df['location_name'].str.extract(r'^(.*) \((.*)\)$')

    crd_df = crd_df[['County', 'State', 'crd_mortality_rate', 'year']].copy()
    crd_df['County'] = crd_df['County'].str.strip()
    crd_df['State'] = crd_df['State'].str.strip()

    merged_df = pd.merge(
        crd_df,
        fips_df[['State_FIPS', 'County_FIPS', 'name', 'state_full', 'fips']],
        left_on=['State', 'County'],
        right_on=['state_full', 'name'],
        how='left'
    )

    merged_df = merged_df.rename(columns={'fips': 'Fips'})

    return merged_df.drop(columns=['name', 'state_full'])

## Test with a Single Year

In [ ]:
df_test = preprocess_fips_crd(2012)
print(f"Shape: {df_test.shape}")
print(f"Columns: {df_test.columns.tolist()}")
df_test.head()

In [ ]:
missing_fips = df_test[df_test['Fips'].isna()]
print(f"Rows with missing FIPS: {len(missing_fips)}")
if len(missing_fips) > 0:
    print("Sample missing rows:")
    print(missing_fips[['County', 'State']].head(10))

## Process All Years (2012-2019)

In [ ]:
output_dir = '../data/processed/preprocessed_fips_crd/'
os.makedirs(output_dir, exist_ok=True)

for year in range(2012, 2020):
    df = preprocess_fips_crd(year)
    df.dropna(inplace=True)
    output_path = f'{output_dir}preprocessed_crd_fips_{year}.csv'
    df.to_csv(output_path, index=False)
    print(f"Year {year}: {len(df)} rows saved to {output_path}")

## Verify Output Files

In [ ]:
files = sorted([f for f in os.listdir(output_dir) if f.endswith('.csv')])
print("Generated files:")
for f in files:
    filepath = os.path.join(output_dir, f)
    df = pd.read_csv(filepath)
    print(f"  {f}: {len(df)} rows, columns: {df.columns.tolist()}")

In [ ]:
df_sample = pd.read_csv(f'{output_dir}preprocessed_crd_fips_2012.csv')
df_sample.head(10)